# Analisis Sentimen Pembangunan Kota Padang 2026

**Tahap 1 — Dataset, validasi, preprocessing, dan perbandingan model.**

Periode awal: **5 September–5 November 2026**. Platform: TikTok, Instagram, Facebook, YouTube, dan X. Notebook ini mengolah data publik yang dikumpulkan melalui API resmi/akses yang diizinkan atau pencatatan manual. Notebook ini tidak melewati login, CAPTCHA, atau pembatasan platform.

Gunakan data asli. Label pelatihan harus diberikan dan diperiksa oleh manusia; jangan menganggap prediksi otomatis sebagai label kebenaran.


## 1. Skema dataset
Satu baris mewakili satu unggahan, komentar, atau balasan. `record_id` adalah ID unik baris; `content_id` adalah ID unggahan/video induk.

Label `sentiment`: `positif`, `negatif`, `netral`, `campuran`, atau `tidak_jelas`. Label `tidak_jelas` tidak digunakan untuk pelatihan model.


In [ ]:
# 2. Import library
import re, html, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix, ConfusionMatrixDisplay)

RANDOM_STATE=42
START_DATE='2026-09-05'
END_DATE='2026-11-05'
pd.set_option('display.max_colwidth',180)
warnings.filterwarnings('ignore')


## 3. Unggah dataset
Unduh `data/padang_social_media_template.csv`, isi dengan data yang dikumpulkan, lalu unggah file CSV tersebut ke Colab.


In [ ]:
try:
    from google.colab import files
    uploaded=files.upload()
    DATA_PATH=next(iter(uploaded.keys())) if uploaded else 'padang_social_media.csv'
except ImportError:
    candidates=['data/padang_social_media.csv','padang_social_media.csv','../data/padang_social_media.csv']
    DATA_PATH=next((c for c in candidates if Path(c).exists()), 'data/padang_social_media.csv')
print('File:',DATA_PATH)


In [ ]:
df=pd.read_csv(DATA_PATH,encoding='utf-8-sig')
df.columns=[c.strip().lower() for c in df.columns]
required=['record_id','platform','content_type','published_at','text','url','sentiment','project_id']
missing=[c for c in required if c not in df.columns]
if missing: raise ValueError(f'Kolom wajib belum tersedia: {missing}')
optional={'content_id':'','parent_content_id':'','collected_at':'','keyword':'','project_name':'','location':'','like_count':0,'reply_count':0,'view_count':0,'share_count':0,'stance':'','issue_theme':'','include':'yes','annotation_note':''}
for c,v in optional.items():
    if c not in df.columns: df[c]=v
print('Jumlah baris:',len(df))
display(df.head())


## 4. Validasi, filter periode, dan deduplikasi
Periode inklusif 5 September sampai 5 November 2026. Kata negasi seperti `tidak`, `bukan`, `belum`, `jangan`, `gak`, dan `nggak` akan dipertahankan pada preprocessing.


In [ ]:
string_cols=['record_id','platform','content_type','content_id','parent_content_id','keyword','project_id','project_name','location','text','url','sentiment','stance','issue_theme','include','annotation_note']
for c in string_cols: df[c]=df[c].fillna('').astype(str).str.strip()
df['platform']=df['platform'].str.lower()
df['sentiment']=df['sentiment'].str.lower()
df['include']=df['include'].str.lower()
df['published_at']=pd.to_datetime(df['published_at'],errors='coerce')
df['collected_at']=pd.to_datetime(df['collected_at'],errors='coerce')
for c in ['like_count','reply_count','view_count','share_count']:
    df[c]=pd.to_numeric(df[c],errors='coerce').fillna(0)
start=pd.Timestamp(START_DATE); end=pd.Timestamp(END_DATE)+pd.Timedelta(days=1)
df=df[df['published_at'].ge(start)&df['published_at'].lt(end)].copy()
df=df[df['include'].isin(['yes','ya','1','true']) & df['text'].str.strip().ne('')].copy()
df['text_dedup']=df['text'].str.lower().str.replace(r'\s+',' ',regex=True).str.strip()
has_id=df['record_id'].ne('')
a=df[has_id].drop_duplicates(['platform','record_id'])
b=df[~has_id].drop_duplicates(['platform','content_id','text_dedup','published_at'])
df=pd.concat([a,b],ignore_index=True).drop(columns=['text_dedup'])
print('Data valid setelah filter:',len(df))
print('Per platform:'); display(df['platform'].value_counts().to_frame('jumlah'))


## 5. Preprocessing teks & Normalisasi Dialek Minang
URL dan mention dihapus; hashtag dipertahankan sebagai kata. Emoji tidak dihapus secara agresif karena dapat membawa informasi sentimen. Kosakata dan dialek lokal Minang dinormalisasi ke padanan bahasa Indonesia agar model TF-IDF dapat mengenali polaritas kata kunci dengan tepat.


In [ ]:
MINANG_NORMALIZATION = {
    r'\bindak\b': 'tidak',
    r'\bndak\b': 'tidak',
    r'\balun\b': 'belum',
    r'\bjan\b': 'jangan',
    r'\brancak bana\b': 'sangat bagus',
    r'\brancak\b': 'bagus',
    r'\btasumbek\b': 'tersumbat',
    r'\btabangkalai\b': 'mangkrak',
    r'\bsarok\b': 'sampah',
    r'\bbuek\b': 'buat',
    r'\baia\b': 'air',
    r'\bmambana\b': 'memohon',
    r'\bpabaelok\b': 'perbaiki',
    r'\bcapek\b': 'cepat',
    r'\bmokasi\b': 'terima kasih',
    r'\bbarasiah\b': 'bersih',
    r'\blalok\b': 'tidur',
    r'\bmanumpuak\b': 'menumpuk',
    r'\bko\b': 'ini',
    r'\btu\b': 'itu',
    r'\bse\b': 'saja',
    r'\btiok\b': 'setiap'
}

def clean_text(text):
    text=html.unescape(str(text)).lower()
    text=re.sub(r'https?://\S+|www\.\S+',' ',text)
    text=re.sub(r'@\w+',' ',text)
    text=re.sub(r'#(\w+)',r'\1',text)
    text=re.sub(r'<[^>]+>',' ',text)
    for pattern, replacement in MINANG_NORMALIZATION.items():
        text=re.sub(pattern, replacement, text)
    return re.sub(r'\s+',' ',text).strip()
df['clean_text']=df['text'].apply(clean_text)
display(df[['text','clean_text','sentiment']].head(10))


## 6. Distribusi label
Pastikan anotasi manusia sudah dilakukan. Model hanya dilatih menggunakan label `positif`, `negatif`, `netral`, dan `campuran`.


In [ ]:
valid_labels=['positif','negatif','netral','campuran']
labeled=df[df['sentiment'].isin(valid_labels)].copy()
counts=labeled['sentiment'].value_counts().reindex(valid_labels,fill_value=0)
display(counts.to_frame('jumlah'))
counts.plot(kind='bar',figsize=(8,4),title='Distribusi Label Sentimen')
plt.xlabel('Sentimen'); plt.ylabel('Jumlah data'); plt.xticks(rotation=0); plt.tight_layout(); plt.show()
if labeled.empty: raise ValueError('Belum ada data berlabel. Isi kolom sentiment terlebih dahulu.')
if labeled['sentiment'].nunique()<2: raise ValueError('Diperlukan minimal dua kelas sentimen.')


## 7. Perbandingan Random Forest, Linear SVM, dan Multinomial Naive Bayes
TF-IDF ditempatkan di dalam `Pipeline` untuk menghindari kebocoran data dari data uji ke proses pelatihan. Pembagian data menggunakan stratifikasi.


In [ ]:
class_counts=labeled['sentiment'].value_counts()
if class_counts.min()<2: raise ValueError('Setiap kelas perlu minimal dua contoh. Tambahkan anotasi atau evaluasi kembali skema kelas.')
X=labeled['clean_text']; y=labeled['sentiment']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.20,random_state=RANDOM_STATE,stratify=y)
models={
 'Random Forest':RandomForestClassifier(n_estimators=300,random_state=RANDOM_STATE,class_weight='balanced',n_jobs=-1),
 'Linear SVM':LinearSVC(class_weight='balanced',random_state=RANDOM_STATE),
 'Multinomial Naive Bayes':MultinomialNB()
}
results=[]; fitted_models={}; predictions={}
for name,clf in models.items():
    pipe=Pipeline([('tfidf',TfidfVectorizer(ngram_range=(1,2),min_df=2,max_features=30000,sublinear_tf=True)),('classifier',clf)])
    pipe.fit(X_train,y_train); pred=pipe.predict(X_test)
    p,r,f,_=precision_recall_fscore_support(y_test,pred,average='macro',zero_division=0)
    results.append({'model':name,'accuracy':accuracy_score(y_test,pred),'precision_macro':p,'recall_macro':r,'f1_macro':f})
    fitted_models[name]=pipe; predictions[name]=pred
results_df=pd.DataFrame(results).sort_values('f1_macro',ascending=False)
display(results_df)


In [ ]:
for name,pred in predictions.items():
    print('\n'+'='*60+'\n'+name)
    print(classification_report(y_test,pred,zero_division=0))
    labels=sorted(y.unique())
    cm=confusion_matrix(y_test,pred,labels=labels)
    ConfusionMatrixDisplay(confusion_matrix=cm,display_labels=labels).plot(xticks_rotation=35,values_format='d')
    plt.title(f'Confusion Matrix — {name}'); plt.tight_layout(); plt.show()


## 8. Ekspor hasil
Simpan dataset bersih, tabel perbandingan model, dan prediksi data uji. Jaga keamanan data sumber; jangan publikasikan informasi pribadi.


In [ ]:
out=Path('outputs'); out.mkdir(exist_ok=True)
best_name=results_df.iloc[0]['model']; best=fitted_models[best_name]
pred_out=pd.DataFrame({'text':X_test.values,'actual_sentiment':y_test.values,'predicted_sentiment':best.predict(X_test)})
df.to_csv(out/'padang_social_media_clean.csv',index=False,encoding='utf-8-sig')
results_df.to_csv(out/'model_comparison.csv',index=False,encoding='utf-8-sig')
pred_out.to_csv(out/'test_predictions.csv',index=False,encoding='utf-8-sig')
print('Model dengan macro F1 tertinggi pada data uji:',best_name)
print('Hasil tersimpan di',out.resolve())


## 9. Catatan metodologis
1. Dataset media sosial bukan sampel probabilitas seluruh penduduk Kota Padang.
2. Volume antarplatform tidak otomatis dapat dibandingkan karena akses, pencarian, dan karakter pengguna berbeda.
3. Sentimen berbeda dari sikap kebijakan: kritik pada satu aspek proyek tidak selalu berarti menolak seluruh proyek.
4. Label pelatihan harus berasal dari anotasi manusia; dokumentasikan pedoman anotasi dan pemeriksaan konsistensinya.
5. Laporkan jumlah data ditemukan, diakses, disaring, dan dianalisis.
6. Jika kelas tidak seimbang, perhatikan macro F1 dan metrik per kelas, bukan accuracy saja.
7. Jangan mempublikasikan username, data pribadi, atau konten yang tidak boleh dibagikan.
